In [ ]:
!nvidia-smi

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time
server = subprocess.Popen(["ollama", "serve"],
                          stdout=open("ollama.log", "w"),
                          stderr=subprocess.STDOUT)
time.sleep(5)
!tail -5 ollama.log

In [ ]:
!ollama pull qwen2.5:7b
# Optional: smarter but slower (~9 GB)
# !ollama pull qwen2.5:14b

In [ ]:
import requests
r = requests.post("http://localhost:11434/api/generate", json={
    "model": "qwen2.5:7b",
    "prompt": "Explain MQTT QoS levels in 3 lines",
    "stream": False
}, timeout=300)
print(r.json()["response"])

In [ ]:
!ollama ps
!grep -iE "cuda|gpu|compute|vram" ollama.log | tail -15

In [ ]:
import requests

def ask(prompt, model="qwen2.5:7b", system=None):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    r = requests.post("http://localhost:11434/api/chat",
                      json={"model": model, "messages": msgs, "stream": False},
                      timeout=600)
    return r.json()["message"]["content"]

print(ask("Write a Python function to compute Modbus RTU CRC16"))

In [ ]:
!ollama pull qwen2.5:14b

import requests, pandas as pd

URL = "http://localhost:11434"
MODELS = ["qwen2.5:7b", "qwen2.5:14b"]
PROMPTS = [
    "Explain how MQTT QoS 1 works in 5 sentences.",
    "Write a Python function that computes Modbus RTU CRC16.",
    "List 5 advantages of edge AI for industrial fire detection.",
]

def gen(model, prompt, **extra):
    return requests.post(f"{URL}/api/generate", timeout=900, json={
        "model": model, "prompt": prompt, "stream": False,
        "options": {"temperature": 0}, **extra}).json()

rows = []
for m in MODELS:
    gen(m, "hi")                                   # warm-up: load into GPU
    ps = requests.get(f"{URL}/api/ps").json()["models"]
    vram = next(x["size_vram"] for x in ps if m in (x.get("name"), x.get("model"))) / 1e9
    for p in PROMPTS:
        r = gen(m, p)
        rows.append({
            "model": m,
            "prompt": p[:35] + "...",
            "gen_tok_per_s": r["eval_count"] / (r["eval_duration"] / 1e9),
            "gen_tokens": r["eval_count"],
            "total_s": r["total_duration"] / 1e9,
            "vram_gb": vram,
        })
    gen(m, "", keep_alive=0)                      # unload before next model

df = pd.DataFrame(rows).round(1)
display(df)

summary = df.groupby("model")[["gen_tok_per_s", "total_s", "vram_gb"]].mean().round(1)
print(summary.to_markdown())